## Model Training - Random Forest Classifier

In [1]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import ydf
import matplotlib.pyplot as plt
# read data
debris_df = pd.read_csv("../Data/DS_combined_final.csv")
debris_df["LIFETIME_CLASS"] = debris_df["LIFETIME_CLASS"].astype("category")

In [2]:
# split the data into training and test sets (80% training, 20% test)
train_df, test_df = train_test_split(debris_df, test_size=0.2, random_state=42)

print("Training set size:", len(train_df))
print("Test set size:", len(test_df))

Training set size: 19377
Test set size: 4845


#### Model 1 - using all features

In [3]:
# train model
# not use columns that equal the label (-> if decay and launch both kept as features) or are unneccessary
features1 = ['OBJECT_TYPE', 'COUNTRY', 'SITE', 'LAUNCH', 
            'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model1 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features = features1).train(train_df, verbose = 2)

Train model on 19377 examples
Model trained in 0:00:14.539693


In [4]:
model1.describe() # show model description

In [5]:
# Evaluate the model
evaluation1 = model1.evaluate(test_df)
print("Accuracy:", evaluation1.accuracy)

Accuracy: 0.952734778121775


In [6]:
evaluation1

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1612,39,16
Class 2,25,1532,92
Class 3,1,56,1472


In [7]:
model1.analyze(test_df) # analyze the test dataset

In [8]:
# save model
model1.save("../Models/RF_Model1")

In [10]:
# load model 1
model1 = ydf.load_model("../Models/RF_Model1")

#### Model 2 - using all features except REV_AT_EPOCH and LAUNCH 
##### They are basically measures for the age of the object and thus are highly correlated with the lifetime. That is beceause for every object the latest OMM file is used where the Epoch is mostly very near the decay date.

In [9]:
features2 = ['OBJECT_TYPE', 'COUNTRY', 'SITE',
            'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model2 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features2).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19377 examples
Model trained in 0:00:03.397081


In [10]:
model2.describe() # show model description

In [11]:
# Evaluate the model
evaluation2 = model2.evaluate(test_df)
print("Accuracy:", evaluation2.accuracy)

Accuracy: 0.8557275541795666


In [12]:
evaluation2

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1464,155,33
Class 2,134,1336,201
Class 3,40,136,1346


In [13]:
# analyze model
model2.analyze(test_df) # analyze the test dataset

In [14]:
# save model
model2.save("../Models/RF_Model2")

In [13]:
# load model
model2 = ydf.load_model("../Models/RF_Model2")

#### Model 3 - using only umerical features except REV_AT_EPOCH

In [15]:
features3 = ['PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model3 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features3).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19377 examples
Model trained in 0:00:03.822024


In [16]:
# model description
model3.describe() # show model description


In [17]:
# Evaluate the model
evaluation3 = model3.evaluate(test_df)
print("Accuracy:", evaluation3.accuracy)

Accuracy: 0.8179566563467492


In [18]:
evaluation3

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1437,180,42
Class 2,155,1249,261
Class 3,46,198,1277


In [19]:
# analyze model
model3.analyze(test_df) 

In [20]:
# save model
model3.save("../Models/RF_Model3")

In [16]:
# load model
model3 = ydf.load_model("../Models/RF_Model3")

#### Model 4 - using normalized numerical features except REV_AT_EPOCH

In [21]:
# split the data into training and test sets (80% training, 20% test)
train_df_scaled, test_df_scaled = train_test_split(debris_df, test_size=0.2, random_state=42)

print("Training set size:", len(train_df_scaled))
print("Test set size:", len(test_df_scaled))

Training set size: 19377
Test set size: 4845


In [22]:
# normalize features using z normalization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# scale all columns except the target column
train_df_scaled.iloc[:, 8:-1] = scaler.fit_transform(train_df_scaled.iloc[:, 8:-1])
test_df_scaled.iloc[:, 8:-1] = scaler.transform(test_df_scaled.iloc[:, 8:-1])

In [23]:
# train model
features4 = ['PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model4 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features4).train(train_df_scaled, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19377 examples
Model trained in 0:00:03.565942


In [24]:
# see model stats
model4.describe()

In [25]:
# Evaluate the model
evaluation4 = model4.evaluate(test_df_scaled)
print("Accuracy:", evaluation4.accuracy)

Accuracy: 0.8196078431372549


In [26]:
evaluation4

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1440,182,39
Class 2,151,1253,263
Class 3,47,192,1278


In [27]:
# analyze model
model4.analyze(test_df_scaled)

In [28]:
# save model
model4.save("../Models/RF_Model4")

#### Model 5 - using only Keplerian Elements

In [29]:
features5 = ['ARG_OF_PERICENTER', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'SEMIMAJOR_AXIS']
model5 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features5).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19377 examples
Model trained in 0:00:03.030000


In [30]:
# see model stats
model5.describe()

In [31]:
# Evaluate the model
evaluation5 = model5.evaluate(test_df)
print("Accuracy:", evaluation5.accuracy)

Accuracy: 0.773374613003096


In [32]:
evaluation5

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1373,194,68
Class 2,186,1166,304
Class 3,79,267,1208


In [33]:
# analyze model
model5.analyze(test_df)

In [34]:
# save model
model5.save("../Models/RF_Model5")

In [22]:
# load model
model5 = ydf.load_model("../Models/RF_Model5")